# 06. PhoBERT Results

Tổng hợp kết quả fine-tuning PhoBERT và robustness trên noisy test.

## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

print("Project root:", ROOT)

TABLES_DIR = ROOT / "outputs" / "tables"
FIGURES_DIR = ROOT / "outputs" / "figures"
PREDICTIONS_DIR = ROOT / "outputs" / "predictions"
REPORTS_DIR = ROOT / "outputs" / "reports"
MODELS_DIR = ROOT / "outputs" / "models" / "phobert"

## 2. Kiểm tra file Stage 6 bắt buộc

Các file này phải tồn tại sau khi tải và giải nén `phobert_stage6_outputs.zip` từ Kaggle về project local.

In [ ]:
required_files = {
    "phobert_clean_results": TABLES_DIR / "phobert_clean_results.csv",
    "phobert_training_history": TABLES_DIR / "phobert_training_history.csv",
    "phobert_clean_classification_report": TABLES_DIR / "phobert_clean_classification_report.csv",
    "phobert_robustness_results": TABLES_DIR / "phobert_robustness_results.csv",
    "phobert_robustness_drop": TABLES_DIR / "phobert_robustness_drop.csv",
    "phobert_robustness_class_report": TABLES_DIR / "phobert_robustness_class_report.csv",
    "phobert_clean_predictions": PREDICTIONS_DIR / "phobert_clean_predictions.csv",
    "phobert_robustness_predictions": PREDICTIONS_DIR / "phobert_robustness_predictions.csv",
    "sentiment_best_model_dir": MODELS_DIR / "sentiment" / "best",
    "topic_best_model_dir": MODELS_DIR / "topic" / "best",
}

check_df = pd.DataFrame(
    [{"name": name, "path": str(path), "exists": path.exists()} for name, path in required_files.items()]
)

display(check_df)

missing = check_df.loc[~check_df["exists"], "name"].tolist()
if missing:
    raise FileNotFoundError(f"Missing required Stage 6 files/folders: {missing}")

print("All required Stage 6 files exist.")

## 3. Load kết quả PhoBERT

In [ ]:
clean_results = pd.read_csv(TABLES_DIR / "phobert_clean_results.csv")
training_history = pd.read_csv(TABLES_DIR / "phobert_training_history.csv")
clean_report = pd.read_csv(TABLES_DIR / "phobert_clean_classification_report.csv")
robustness_results = pd.read_csv(TABLES_DIR / "phobert_robustness_results.csv")
robustness_drop = pd.read_csv(TABLES_DIR / "phobert_robustness_drop.csv")
robustness_class_report = pd.read_csv(TABLES_DIR / "phobert_robustness_class_report.csv")

display(Markdown("### Clean results"))
display(clean_results)

display(Markdown("### Training history"))
display(training_history)

display(Markdown("### Robustness results"))
display(robustness_results)

display(Markdown("### Robustness drop"))
display(robustness_drop)

## 4. Clean test results

Nội dung: xác nhận PhoBERT đạt hiệu năng in-distribution trên dữ liệu test sạch.

In [ ]:
clean_view = clean_results[
    ["task", "model", "variant", "accuracy", "macro_f1", "weighted_f1", "best_epoch", "best_val_macro_f1", "train_seconds"]
].copy()

display(clean_view)

for _, row in clean_view.iterrows():
    display(Markdown(
        f"- **{row['task']}**: Macro-F1 = **{row['macro_f1']:.4f}**, "
        f"Accuracy = **{row['accuracy']:.4f}**, best_epoch = **{int(row['best_epoch'])}**"
    ))

## 5. So sánh PhoBERT với baseline trên clean test

Nội dung: kiểm tra PhoBERT có vượt baseline truyền thống trên dữ liệu sạch hay không.

Baseline được lấy từ:

```text
outputs/tables/baseline_results.csv
```

In [ ]:
baseline_results_path = TABLES_DIR / "baseline_results.csv"

if not baseline_results_path.exists():
    display(Markdown("⚠️ Missing `baseline_results.csv`. Skip clean baseline comparison."))
else:
    baseline_results = pd.read_csv(baseline_results_path)
    baseline_clean = baseline_results[baseline_results["split"] == "test"].copy()

    best_baseline_clean = (
        baseline_clean.sort_values(["task", "macro_f1"], ascending=[True, False])
        .groupby("task")
        .head(1)
        .reset_index(drop=True)
    )

    phobert_clean = clean_results[["task", "accuracy", "macro_f1", "weighted_f1"]].copy()
    phobert_clean = phobert_clean.rename(
        columns={
            "accuracy": "phobert_accuracy",
            "macro_f1": "phobert_macro_f1",
            "weighted_f1": "phobert_weighted_f1",
        }
    )

    compare_clean = best_baseline_clean[
        ["task", "model", "accuracy", "macro_f1", "weighted_f1"]
    ].rename(
        columns={
            "model": "best_baseline_model",
            "accuracy": "baseline_accuracy",
            "macro_f1": "baseline_macro_f1",
            "weighted_f1": "baseline_weighted_f1",
        }
    ).merge(phobert_clean, on="task", how="inner")

    compare_clean["macro_f1_gain"] = compare_clean["phobert_macro_f1"] - compare_clean["baseline_macro_f1"]
    compare_clean["accuracy_gain"] = compare_clean["phobert_accuracy"] - compare_clean["baseline_accuracy"]

    display(compare_clean)

    for _, row in compare_clean.iterrows():
        display(Markdown(
            f"- **{row['task']}**: PhoBERT Macro-F1 = **{row['phobert_macro_f1']:.4f}**, "
            f"best baseline `{row['best_baseline_model']}` Macro-F1 = **{row['baseline_macro_f1']:.4f}**, "
            f"gain = **{row['macro_f1_gain']:.4f}**."
        ))

## 6. Training history

Nội dung: kiểm tra quá trình fine-tuning có ổn định không và epoch tốt nhất có hợp lý không.

In [ ]:
display(training_history)

for task, group_df in training_history.groupby("task"):
    display(Markdown(f"### Training history — {task}"))
    display(group_df.sort_values("epoch"))

    best_row = group_df.sort_values("val_macro_f1", ascending=False).iloc[0]
    display(Markdown(
        f"Best validation Macro-F1 for **{task}**: **{best_row['val_macro_f1']:.4f}** "
        f"at epoch **{int(best_row['epoch'])}**."
    ))

### Training curve figures

In [ ]:
for task in ["sentiment", "topic"]:
    path = FIGURES_DIR / f"phobert_training_curve_{task}.png"
    if path.exists():
        display(Markdown(f"#### {task}"))
        display(Image(filename=str(path)))
    else:
        display(Markdown(f"⚠️ Missing figure: `{path}`"))

## 7. Clean per-class analysis

Nội dung: xem lớp nào còn yếu trên clean test.

In [ ]:
summary_labels = {"accuracy", "macro avg", "weighted avg"}

clean_per_class = clean_report[
    ~clean_report["label"].isin(summary_labels)
].copy()

display(clean_per_class.sort_values(["task", "f1_score"]))

for task, group_df in clean_per_class.groupby("task"):
    weakest = group_df.sort_values("f1_score").iloc[0]
    display(Markdown(
        f"- Weakest clean class for **{task}**: `{weakest['label']}` "
        f"with F1 = **{weakest['f1_score']:.4f}**."
    ))

## 8. Clean confusion matrices

In [ ]:
for task in ["sentiment", "topic"]:
    path = FIGURES_DIR / f"confusion_matrix_phobert_clean_{task}.png"
    if path.exists():
        display(Markdown(f"### PhoBERT clean confusion matrix — {task}"))
        display(Image(filename=str(path)))
    else:
        display(Markdown(f"⚠️ Missing confusion matrix: `{path}`"))

## 9. Robustness results

Nội dung: xem PhoBERT suy giảm bao nhiêu trên từng loại noise.

In [ ]:
variant_order = [
    "clean",
    "typo_light",
    "typo_medium",
    "teencode_light",
    "mixed_light",
    "no_accent",
    "mixed_no_accent",
]

robustness_view = robustness_results.copy()
robustness_view["variant"] = pd.Categorical(
    robustness_view["variant"],
    categories=variant_order,
    ordered=True,
)
robustness_view = robustness_view.sort_values(["task", "variant"])

display(robustness_view)

## 10. Robustness drop

Nội dung: xác định loại nhiễu gây giảm Macro-F1 mạnh nhất.

In [ ]:
drop_view = robustness_drop.copy()
drop_view["variant"] = pd.Categorical(
    drop_view["variant"],
    categories=variant_order,
    ordered=True,
)
drop_view = drop_view.sort_values(["task", "variant"])

display(drop_view)

worst_drop = (
    robustness_drop[robustness_drop["variant"] != "clean"]
    .sort_values(["task", "macro_f1_drop"], ascending=[True, False])
    .groupby("task")
    .head(1)
    .reset_index(drop=True)
)

display(Markdown("### Worst PhoBERT robustness drop by task"))
display(worst_drop[[
    "task",
    "variant",
    "clean_macro_f1",
    "variant_macro_f1",
    "macro_f1_drop",
    "macro_f1_relative_drop_percent",
]])

for _, row in worst_drop.iterrows():
    display(Markdown(
        f"- **{row['task']}**: worst variant = `{row['variant']}`, "
        f"Macro-F1 drop = **{row['macro_f1_drop']:.4f}** "
        f"({row['macro_f1_relative_drop_percent']:.2f}%)."
    ))

## 11. So sánh PhoBERT với baseline trên noisy variants

Nội dung: kiểm tra PhoBERT có bền hơn baseline truyền thống trên từng variant hay không.

Baseline robustness được lấy từ:

```text
outputs/tables/baseline_robustness_results.csv
```

In [ ]:
baseline_robustness_path = TABLES_DIR / "baseline_robustness_results.csv"

if not baseline_robustness_path.exists():
    display(Markdown("⚠️ Missing `baseline_robustness_results.csv`. Skip noisy baseline comparison."))
else:
    baseline_robustness = pd.read_csv(baseline_robustness_path)

    best_baseline_by_variant = (
        baseline_robustness
        .sort_values(["task", "variant", "macro_f1"], ascending=[True, True, False])
        .groupby(["task", "variant"])
        .head(1)
        .reset_index(drop=True)
    )

    phobert_by_variant = robustness_results[
        ["task", "variant", "accuracy", "macro_f1", "weighted_f1"]
    ].rename(
        columns={
            "accuracy": "phobert_accuracy",
            "macro_f1": "phobert_macro_f1",
            "weighted_f1": "phobert_weighted_f1",
        }
    )

    compare_noisy = best_baseline_by_variant[
        ["task", "variant", "model", "accuracy", "macro_f1", "weighted_f1"]
    ].rename(
        columns={
            "model": "best_baseline_model",
            "accuracy": "baseline_accuracy",
            "macro_f1": "baseline_macro_f1",
            "weighted_f1": "baseline_weighted_f1",
        }
    ).merge(phobert_by_variant, on=["task", "variant"], how="inner")

    compare_noisy["macro_f1_gain"] = compare_noisy["phobert_macro_f1"] - compare_noisy["baseline_macro_f1"]
    compare_noisy["accuracy_gain"] = compare_noisy["phobert_accuracy"] - compare_noisy["baseline_accuracy"]
    compare_noisy["variant"] = pd.Categorical(compare_noisy["variant"], categories=variant_order, ordered=True)
    compare_noisy = compare_noisy.sort_values(["task", "variant"])

    display(compare_noisy)

    display(Markdown("### Variants where PhoBERT is worse than best baseline"))
    worse_than_baseline = compare_noisy[compare_noisy["macro_f1_gain"] < 0].copy()
    display(worse_than_baseline[[
        "task",
        "variant",
        "best_baseline_model",
        "baseline_macro_f1",
        "phobert_macro_f1",
        "macro_f1_gain",
    ]])

## 12. Per-class robustness: clean vs no_accent

Nội dung: kiểm tra lớp nào sụp mạnh nhất khi bỏ dấu.

In [ ]:
per_class = robustness_class_report[
    ~robustness_class_report["label"].isin(summary_labels)
].copy()

clean_pc = per_class[per_class["variant"] == "clean"][
    ["task", "label", "precision", "recall", "f1_score"]
].rename(
    columns={
        "precision": "clean_precision",
        "recall": "clean_recall",
        "f1_score": "clean_f1",
    }
)

no_accent_pc = per_class[per_class["variant"] == "no_accent"][
    ["task", "label", "precision", "recall", "f1_score"]
].rename(
    columns={
        "precision": "no_accent_precision",
        "recall": "no_accent_recall",
        "f1_score": "no_accent_f1",
    }
)

pc_drop = clean_pc.merge(no_accent_pc, on=["task", "label"], how="inner")
pc_drop["f1_drop"] = pc_drop["clean_f1"] - pc_drop["no_accent_f1"]
pc_drop["relative_f1_drop_percent"] = pc_drop.apply(
    lambda row: 0.0 if row["clean_f1"] == 0 else round(row["f1_drop"] / row["clean_f1"] * 100, 4),
    axis=1,
)

pc_drop = pc_drop.sort_values(["task", "f1_drop"], ascending=[True, False])

display(pc_drop)

for task, group_df in pc_drop.groupby("task"):
    worst_class = group_df.sort_values("f1_drop", ascending=False).iloc[0]
    display(Markdown(
        f"- **{task}**: largest F1 drop under no_accent is `{worst_class['label']}` "
        f"with drop = **{worst_class['f1_drop']:.4f}**."
    ))

## 13. Per-class robustness across all variants

Nội dung: xem F1 của từng lớp biến động theo các loại noise.

In [ ]:
per_class_view = per_class.copy()
per_class_view["variant"] = pd.Categorical(
    per_class_view["variant"],
    categories=variant_order,
    ordered=True,
)
per_class_view = per_class_view.sort_values(["task", "label", "variant"])

for task in per_class_view["task"].unique():
    display(Markdown(f"### {task}"))
    task_df = per_class_view[per_class_view["task"] == task]
    pivot = task_df.pivot_table(
        index="variant",
        columns="label",
        values="f1_score",
        observed=False,
    ).reindex(variant_order)
    display(pivot)

## 14. Kết luận Stage 6

Kết luận cần ghi nhận:

```text
1. PhoBERT vượt baseline truyền thống trên clean test ở cả sentiment và topic.
2. PhoBERT cải thiện rõ các lớp khó trên clean test, đặc biệt neutral và others.
3. PhoBERT ổn định với typo_light, typo_medium, teencode_light và mixed_light.
4. PhoBERT suy giảm rất mạnh trên no_accent và mixed_no_accent.
5. Trên no_accent, PhoBERT kém hơn TF-IDF char SVM ở cả hai task.
6. Kết quả này cho thấy PhoBERT không tự động bền vững với tiếng Việt mất dấu.
```

Diễn giải học thuật an toàn:

```text
Clean test đánh giá hiệu năng in-distribution.
Noisy test đánh giá robustness dưới nhiễu có kiểm soát.
Các noisy variants không được dùng để train hoặc chọn checkpoint.
```

In [ ]:
display(Markdown("### Final PhoBERT clean results"))
display(clean_results)

display(Markdown("### Final PhoBERT worst drops"))
display(worst_drop[[
    "task",
    "variant",
    "clean_macro_f1",
    "variant_macro_f1",
    "macro_f1_drop",
    "macro_f1_relative_drop_percent",
]])